<center><h1>Companhia Aberta Demonstrativo Financeiro</h1></center>

# <h2>Load Libraries</h2>

In [15]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 10)

# <h2>Import Data</h2>

## <h3>Load .csv</h3>

In [16]:
cia_aberta_df = pd.read_csv(r'dfp_cia_aberta_BPA_con_2020.csv', encoding = 'ISO-8859-1', sep=";")

## <h3>Select Company</h3>

In [17]:
df = cia_aberta_df[cia_aberta_df['CNPJ_CIA'] == '97.837.181/0001-47'].copy()    # (seleciono somente as linhas relativas a uma companhia de interesse)
df = df[['ORDEM_EXERC','CD_CONTA','DS_CONTA','VL_CONTA']]
df = df[df['ORDEM_EXERC'] == 'ÚLTIMO']                            # seleciono somente o último ou penúltimo exercício
df.reset_index(inplace=True, drop=True)

In [18]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA
0,ÚLTIMO,1,Ativo Total,11498520.0
1,ÚLTIMO,1.01,Ativo Circulante,4220022.0
2,ÚLTIMO,1.01.01,Caixa e Equivalentes de Caixa,1728413.0
3,ÚLTIMO,1.01.02,Aplicações Financeiras,0.0
4,ÚLTIMO,1.01.02.01,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...,...
71,ÚLTIMO,1.02.04.02.07,Goodwill na aquisição da Caetex Florestal,8767.0
72,ÚLTIMO,1.02.04.02.08,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
73,ÚLTIMO,1.02.04.02.09,Goodwill na aquisição da Massima Revestimentos...,6110.0
74,ÚLTIMO,1.02.04.02.10,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


# <h2>Wrangling</h2>

## <h3>Number of Steps</h3>

In [19]:
cd_conta_split = pd.Series([string.split('.') for string in df['CD_CONTA']])
cd_conta_len = [len(lst) for lst in cd_conta_split]
num_steps = max(cd_conta_len)

In [20]:
num_steps

5

## <h3>For Loop</h3>

In [21]:
for round in range(1, num_steps+1):

    ## JOIN

    cd_conta_split = pd.Series([string.split('.') for string in df['CD_CONTA']])
    cd_conta_joincol = [row[:round] for row in cd_conta_split]

    # Select rows with n elements, where n=round:
    len_key = [len(row) for row in cd_conta_split]
    min_len = np.array(len_key).min()
    ind_min_len = np.where(len_key==min_len)[0]
    keys = [cd_conta_joincol[ind] for ind in ind_min_len]
    key_idx_dict = {".".join(key): ind for key, ind in zip(keys, ind_min_len)}

    # Join:
    ds_conta = [df['DS_CONTA'][key_idx_dict[key]]
                  for key in key_idx_dict
                  for row in cd_conta_joincol
                  if ".".join(row) == key]
    # Insert:
    colname='DS_CONTA' + '_' + str(round)

    if pd.Series(ds_conta).equals(df['DS_CONTA']):
        df.rename(columns={'DS_CONTA': colname}, inplace=True)
        df.to_csv("output.csv", index=False)
        exit
    else:
        df.insert(round+1, column=colname, value=ds_conta)

    ## Drop rows:

    # For each key which was used to join, check if there is at least another row in CD_CONTA which code starts with the same key
    # (example: CD_CONTA '1.01' starting with '1'; CD_CONTA '1.01.01' starting with '1.01', etc.)
    # If so, drop the row. Else, don't drop

    for ind in ind_min_len:
        mask = [True]*len(ds_conta)
        mask[ind] = False
        if any(row[:min_len] == cd_conta_joincol[ind] for row in cd_conta_split[mask]):
            df = df.drop(ind)
        else:
            df.loc[ind, 'CD_CONTA'] = df.loc[ind, 'CD_CONTA'] + '.00'

    df.reset_index(inplace=True, drop=True)


In [22]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA_4,DS_CONTA_5,VL_CONTA
0,ÚLTIMO,1.01.01.00.00.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02.01.01.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos para Negociação,0.0
2,ÚLTIMO,1.01.02.01.02.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos Designados a Valor Justo,0.0
3,ÚLTIMO,1.01.02.02.00.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,ÚLTIMO,1.01.02.03.00.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
...,...,...,...,...,...,...,...,...
49,ÚLTIMO,1.02.04.02.07.00,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Caetex Florestal,8767.0
50,ÚLTIMO,1.02.04.02.08.00,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
51,ÚLTIMO,1.02.04.02.09.00,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Massima Revestimentos...,6110.0
52,ÚLTIMO,1.02.04.02.10.00,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0
